# Clasificación de textos en Esperanto

**Minería de Textos · CUNEF · 2026/2027**

Clasificamos párrafos en esperanto en tres temas de transporte: **ferrocarril** (`fervojo`),
**automóvil** (`automobilo`) y **aviación** (`aviado`). El cuaderno sigue las tres fases de la práctica:

1. **Dataset**: párrafos de la Wikipedia en esperanto etiquetados por tema.
2. **NLP**: nuestro propio modelo de NLP basado en reglas, con una API parecida a la de spaCy.
3. **Clasificación**: vectorizamos el dataset, entrenamos un modelo y lo probamos con páginas web.

## Instalación de librerías

Todas las librerías que usa el cuaderno, con sus versiones, están en `requirements.txt`. Si el
cuaderno se ejecuta dentro del repositorio se usa el fichero local; si no (por ejemplo en Colab),
se lee de GitHub. En Colab casi todas vienen ya instaladas: la única que falta es `esperantilo`.

In [ ]:
import os

URL_REQUIREMENTS = "https://raw.githubusercontent.com/guilletaff/Clasificacion_Textos_Esperanto/main/requirements.txt"
REQUIREMENTS = "../requirements.txt" if os.path.exists("../requirements.txt") else URL_REQUIREMENTS

%pip install -q -r {REQUIREMENTS}

## 0. Configuración

Con `GENERAR_DATASET = False` se carga el CSV que ya tenemos generado (es más rápido y siempre da
el mismo resultado): se lee de `data/` si el cuaderno está dentro del repositorio y, si no (por
ejemplo en Colab), de GitHub. Con `True` se vuelve a descargar todo de la Wikipedia, se limpia y
se filtra, y el CSV resultante se guarda en `data/`.

In [ ]:
GENERAR_DATASET = True

# Dataset: igual que las stop-words, se lee del disco si existe y si no de GitHub
RUTA_DATASET = "../data/dataset_esperanto.csv"
URL_DATASET = "https://raw.githubusercontent.com/guilletaff/Clasificacion_Textos_Esperanto/main/data/dataset_esperanto.csv"
# Stop-words: fichero resources/vortoj.json de nuestro repositorio. Se lee del disco si el cuaderno
# está dentro del repositorio; en Colab (donde no está el fichero) se lee ese mismo fichero de GitHub
RUTA_STOPWORDS = "../resources/vortoj.json"
URL_STOPWORDS = "https://raw.githubusercontent.com/guilletaff/Clasificacion_Textos_Esperanto/main/resources/vortoj.json"

import os
import re
import json
import sys
import time
from collections import Counter
from functools import lru_cache

import requests
import pandas as pd
import matplotlib.pyplot as plt
import sklearn, bs4

#SEED
SEMILLA = 42
print("Python", sys.version.split()[0], "| pandas", pd.__version__, "| scikit-learn", sklearn.__version__,
      "| requests", requests.__version__, "| beautifulsoup4", bs4.__version__)

# Fase 1 · Dataset

## 1.1 Temas y artículos

Elegimos tres temas de transporte porque comparten bastante vocabulario (`motoro`, `rapido`,
`pasaĝero`, `veturilo`...) y así la clasificación no es trivial. Para cada tema escogimos a mano
artículos de la Wikipedia en esperanto.

Para tener al menos 300 párrafos por clase después de la limpieza, buscamos más artículos en las
categorías de Wikipedia de cada tema (*Kategorio:Fervojo*, *Kategorio:Aŭtomobiloj*,
*Kategorio:Aviado*) y contamos cuántos párrafos válidos aportaba cada uno. Descartamos los títulos
que no existen en esperanto, los que son redirecciones a artículos que ya teníamos (`Stacidomo`) y
los que mezclan temas (`Dizelmotoro` también trata de locomotoras, `Trolebuso` se parece al tranvía).
Tampoco usamos los artículos de las páginas con las que probamos el modelo en la Fase 3
(`Transsiberia fervojo`, `Formulo 1`), para no evaluar con textos que ya ha visto.

In [ ]:
CATEGORIAS = {
    "fervojo": [            # ferrocarril
        "Fervojo", "Lokomotivo", "Metroo", "Tramo", "Rapidtrajno", "Vagono",
        "Fervoja stacio", "Elektra lokomotivo", "Maglevo", "Trajno", "Vaporlokomotivo",
        "Altrapida fervojo", "Funikularo", "Metroo de Moskvo", "Kampa fervojo", "Ŝinkanseno",
        "Koridora fervojo", "Metroo de Londono", "Metroo de Parizo", "Etŝpura fervojo",
        "Trakforko", "Dentorela fervojo", "Kontaktrelo", "Ŝpuro", "Signalilo",
        "Vapora motorvagono", "TGV", "Onda fervojo",
    ],
    "automobilo": [         # automóvil
        "Aŭtomobilo", "Kamiono", "Aŭtobuso", "Elektra aŭtomobilo", "Motociklo",
        "Eksplodmotoro", "Taksio", "Aŭtovojo", "Sportaŭto", "Karoserio", "Aŭtosporto",
        "Tesla Model 3", "Trabant", "Kampadveturilo", "Trafiklumoj", "Messerschmitt Kabinenroller",
        "Aŭtokodo", "BMW Isetta", "Kondukpermesilo", "Benzino", "Kombinaŭto", "Ŝoforo",
        "Toyota", "Ford", "Pneŭmatiko",
    ],
    "aviado": [             # aviación
        "Aviadilo", "Aviado", "Flughaveno", "Helikoptero", "Jetmotoro", "Boeing 747",
        "Flugkompanio", "Aeronaŭtiko", "Piloto", "Airbus", "Concorde", "Aerspaco",
        "Aviadilumo", "Aviadila akcidento", "Paraŝuto", "Aeroplano", "Airbus A380",
        "Senpilota aviadilo", "Nigra skatolo (aviadilo)", "Flugkompania nabo",
        "Programo por oftaj flugantoj", "Aviadila ŝipo", "Kontrolturo",
    ],
}

## 1.2 Descarga de los artículos

Usamos `WikiPage`, de la librería del repositorio guía (`esperantilo`), que se instala con
`requirements.txt`. Si un artículo no se puede descargar, se avisa y se sigue con el resto.

Un detalle importante: si un título no existe en esperanto, `WikiPage` lo busca por defecto en
español e inglés y, si no, en cualquier idioma. En la primera versión del dataset se nos colaron
así párrafos en español (de *Komerca aviado* y *Supersona aviadilo*). Por eso le pasamos
`fallback_languages=()` y `any_language=False`, y comprobamos que la página devuelta esté en
esperanto.

In [ ]:
def descargar_articulo(titulo):
    from esperantilo import WikiPage   # se importa aquí: solo hace falta si se regenera el dataset

    # Solo esperanto: sin idiomas de respaldo ni búsqueda en otros idiomas
    pagina = WikiPage.look_up(titulo, language="eo", fallback_languages=(), any_language=False)
    if pagina.language != "eo":
        raise ValueError(f"solo existe en '{pagina.language}'")
    return pagina.full_text()

## 1.3 Párrafos y limpieza

Cada artículo se corta por saltos de línea (una línea = un párrafo) y cada párrafo se limpia:
quitamos los títulos de sección (`== Historio ==`), las referencias tipo `[1]`, las URLs y el
carácter `;`, que es el separador del CSV.

Cada párrafo hereda la etiqueta de su artículo.

In [ ]:
def limpiar_parrafo(texto):
    texto = re.sub(r"==+.*?==+", "", texto)                # títulos de sección
    texto = re.sub(r"\[\d+\]", "", texto)                  # referencias [1], [2]...
    texto = re.sub(r"https?://\S+|www\.\S+", "", texto)    # enlaces
    texto = texto.replace(";", ",")                        # ';' rompería el CSV
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto


def construir_dataset():
    filas = []
    for categoria, titulos in CATEGORIAS.items():
        print(f"--- {categoria} ---")
        for titulo in titulos:
            try:
                texto = descargar_articulo(titulo)
                parrafos = [limpiar_parrafo(p) for p in texto.split("\n")]
                parrafos = [p for p in parrafos if p]
                filas += [{"text": p, "class": categoria} for p in parrafos]
                print(f"  {titulo}: {len(parrafos)} párrafos")
            except Exception as e:
                # si un artículo falla seguimos con el resto
                print(f"  {titulo}: ERROR ({e})")
            time.sleep(1)   # para no saturar la API de Wikipedia
    return pd.DataFrame(filas, columns=["text", "class"])

## 1.4 Filtrado del ruido

Revisando el CSV vimos que muchos "párrafos" no son texto útil. Los descartamos con reglas
sencillas:

| Motivo | Regla | Ejemplo |
|---|---|---|
| Muy corto | menos de 40 caracteres | títulos sueltos |
| Bibliografía | contiene palabras de cita (`ISBN`, `Arkivigite`, `Arkivita`, `Alirita`, `Wayback`, `Blogger`, `podkasto`...), empieza por `(en)`, `(*)` o una fecha entre paréntesis, o tiene menos de 300 caracteres y una fecha tipo `2018-05-26` | `Stanislavo Belov (2010-10-16) ... Blogger. Arkivita el la originalo ... Alirita ...` |
| Fragmento de lista | empieza en minúscula | `la germana ICE, je kiu baziĝas la dua generacio de AVE` |
| Otro idioma | 2 o más palabras típicas de español, inglés o alemán (`que`, `los`, `the`, `und`, `sich`, `mit`...) | `La aviación de transporte regular comercial es una actividad...` |
| Elemento de lista | menos de 100 caracteres y ningún verbo conjugado (`-as`, `-is`, `-os`, `-us`)* | `Internacia flughaveno Mariupol (Mariupolo)` |
| Duplicado | texto repetido | |

\* Algunos nombres propios también acaban así (`Airbus`, `Julius`, `Venizelos`), y al principio la
regla los tomaba por verbos y dejaba pasar líneas como `Flughaveno Eleftherios Venizelos (Ateno)`.
Ahora una palabra en mayúscula solo cuenta como verbo si le sigue una palabra en minúscula
(`Ekzistas diversaj...`), que es lo que pasa cuando el verbo abre la frase.

La regla de *fragmento de lista* existe porque la Wikipedia corta muchas listas y algunos párrafos
en varias líneas: cada trozo que sigue al primero empieza en minúscula y no tiene sentido por sí
solo. Un párrafo de verdad siempre empieza en mayúscula.

Estas reglas se aplican siempre que se carga el CSV. Además, al regenerar el dataset se guarda ya
filtrado, para que el CSV cumpla el contrato de datos (sin enlaces, referencias ni bibliografía).
Sobre un CSV ya limpio las reglas no descartan nada, así que no importa aplicarlas dos veces.

In [ ]:
LONGITUD_MINIMA = 40
PALABRAS_OTROS_IDIOMAS = {"que", "los", "las", "del", "una", "y", "the", "and", "of", "with",
                          "und", "der", "die", "das", "für", "sie", "sich", "ein", "eine", "im", "mit"}
# Verbo conjugado: palabra en minúscula acabada en -as/-is/-os/-us, o en mayúscula (inicio de
# frase) si le sigue una palabra en minúscula. Así 'Ekzistas diversaj...' cuenta como verbo, pero
# nombres como 'Airbus A380' o 'Julius Nyerere' no
VERBO_CONJUGADO = r"\b[a-zĉĝĥĵŝŭ]\w*(as|is|os|us)\b|\b\w+(as|is|os|us) [a-zĉĝĥĵŝŭ]"


def motivo_descarte(texto):
    # Devuelve por qué se descarta un párrafo, o None si es válido
    if len(texto) < LONGITUD_MINIMA:
        return "muy corto"
    # Bibliografía: palabras típicas de las citas de Wikipedia...
    if re.search(r"ISBN|Arkivigite|Arkivita|Alirita|Wayback|traduko de teksto|Blogger|podkasto", texto):
        return "bibliografía"
    # ... citas que empiezan por idioma "(en)", nota "(*)" o fecha "(27-a de marto..."
    if re.match(r"\(\w{2}\)|\(\*\)|\(\d", texto):
        return "bibliografía"
    # ... y líneas cortas con fechas tipo 2018-05-26
    if re.search(r"\d{4}-\d{2}-\d{2}", texto) and len(texto) < 300:
        return "bibliografía"
    # Un párrafo de verdad empieza en mayúscula: si no, es un trozo de lista o de otro párrafo
    if texto[0].islower():
        return "fragmento de lista"
    palabras = re.findall(r"\w+", texto.lower())
    if sum(p in PALABRAS_OTROS_IDIOMAS for p in palabras) >= 2:
        return "otro idioma"
    if len(texto) < 100 and not re.search(VERBO_CONJUGADO, texto):
        return "elemento de lista"
    return None


def filtrar_dataset(df):
    df = df.copy()
    df["text"] = df["text"].map(limpiar_parrafo)
    df["motivo"] = df["text"].map(motivo_descarte)
    df.loc[df["motivo"].isna() & df["text"].duplicated(), "motivo"] = "duplicado"

    print("Párrafos descartados por clase y motivo:")
    print(pd.crosstab(df["class"], df["motivo"].fillna("válido"), margins=True), "\n")

    return df[df["motivo"].isna()][["text", "class"]].reset_index(drop=True)

## 1.5 Generar o cargar el dataset

Con `GENERAR_DATASET = False` el dataset se busca en este orden: en el disco (`data/`), en nuestro
GitHub y, si los dos fallan, se regenera desde la Wikipedia. Así el cuaderno sobrevive a un fallo de
cualquiera de las webs, como pide el enunciado. Si no se consigue de ninguna forma, se para con un
mensaje que explica qué ha fallado.

In [ ]:
def cargar_dataset_guardado():
    # 1º del disco, 2º de nuestro GitHub, 3º regenerándolo desde la Wikipedia
    if os.path.exists(RUTA_DATASET):
        print("Dataset leído de", RUTA_DATASET)
        return pd.read_csv(RUTA_DATASET, sep=";", encoding="utf-8")
    try:
        datos = pd.read_csv(URL_DATASET, sep=";", encoding="utf-8")
        print("Dataset leído de", URL_DATASET)
        return datos
    except Exception as e:
        print(f"No se pudo leer {URL_DATASET} ({type(e).__name__}): se regenera desde la Wikipedia")
    datos = construir_dataset()
    faltan = set(CATEGORIAS) - set(datos["class"])
    if faltan:
        raise RuntimeError("No se pudo obtener el dataset: no está en el disco, GitHub no responde y "
                           f"de la Wikipedia no se descargó ningún artículo de: {', '.join(sorted(faltan))}")
    return datos


generado = False
if GENERAR_DATASET:
    df_bruto = construir_dataset()
    # si falla la descarga de alguna clase entera, usamos el dataset guardado
    generado = df_bruto["class"].nunique() == len(CATEGORIAS)
    if not generado:
        print("Faltan clases en el dataset generado, se usa el guardado")
if not generado:
    df_bruto = cargar_dataset_guardado()

print("Párrafos antes de filtrar:", len(df_bruto))
df = filtrar_dataset(df_bruto)

if generado:
    # Se guarda ya limpio y filtrado, para que el CSV cumpla el contrato de datos
    os.makedirs(os.path.dirname(RUTA_DATASET), exist_ok=True)
    df.to_csv(RUTA_DATASET, sep=";", index=False, encoding="utf-8")
    print("Dataset guardado en", RUTA_DATASET)

# Comprobamos el contrato de datos
assert list(df.columns) == ["text", "class"]
assert not df["text"].str.contains(";").any()

print("Párrafos después de filtrar:", len(df))
print(df["class"].value_counts())
df.head()

**Conclusiones de la Fase 1.** Más de la mitad de las líneas que devuelve la Wikipedia no son
párrafos útiles: al generar el dataset, de 2354 líneas nos quedamos con 981. El CSV se guarda ya
filtrado, así que al cargarlo la tabla de arriba no muestra descartes; estos son los de la
generación:

| Motivo | Líneas descartadas |
|---|---|
| Muy corto (títulos de sección, elementos sueltos, pies de imagen...) | 887 |
| Elemento de lista (por ejemplo, la lista de aeropuertos de *Flughaveno*) | 231 |
| Fragmento de lista (trozos de listas o párrafos cortados, que empiezan en minúscula) | 127 |
| Bibliografía | 112 |
| Otro idioma | 14 |
| Duplicado | 2 |

Además, aunque un párrafo pase los filtros, la etiqueta heredada del
artículo no siempre es correcta: hay frases genéricas ("Ĝi estis fondita en 1950") o que hablan de
otro tema (un párrafo sobre turbinas de gas dentro del artículo de motores de coche). En la
sección 2.9 usamos el modelo de NLP para medir parte de ese ruido.

# Fase 2 · Modelo de NLP para Esperanto

Construimos el modelo desde cero, sin librerías de NLP (sin spaCy, NLTK ni similares). Usamos re para las expresiones regulares y pandas solo para mostrar tablas.
Se basa en reglas porque el esperanto es muy regular: **la terminación de una palabra dice qué
tipo de palabra es**.

| Terminación | Categoría | Ejemplo |
|---|---|---|
| `-o` | sustantivo | `trajno` (tren) |
| `-a` | adjetivo | `rapida` (rápido) |
| `-e` | adverbio | `rapide` (rápidamente) |
| `-i` | verbo en infinitivo | `veturi` (viajar) |
| `-as` / `-is` / `-os` | verbo en presente / pasado / futuro | `veturis` |
| `-us` / `-u` | condicional / imperativo | `veturus` |
| `-j` | plural | `trajnoj` |
| `-n` | acusativo (complemento directo) | `trajnon` |
| `-ant-`, `-int-`, `-ont-`, `-at-`, `-it-`, `-ot-` | participios | `konstruita` (construido) |

Lo único que no sigue estas reglas es un grupo pequeño y cerrado de palabras (artículo,
pronombres, preposiciones, conjunciones...), que guardamos en un **léxico**.

El modelo funciona como spaCy: `doc = nlp(texto)` devuelve un `Doc` formado por `Token`s, y cada
token tiene `text`, `lemma_`, `pos_`, `morph` e `is_stop`. Para el POS usamos las etiquetas
universales que usa spaCy (`NOUN`, `VERB`, `ADJ`...).

## 2.1 Léxico de palabras cerradas

Guardamos cada palabra con su categoría y su lema. Los pronombres y la tabla de correlativos
(`ki-`, `ti-`, `i-`, `ĉi-`, `neni-` combinados con `-u`, `-o`, `-a`, `-e`, `-am`...) los
generamos con bucles en lugar de escribir todas sus formas (`kiu`, `kiuj`, `kiun`, `kiujn`...).

In [ ]:
PRONOMBRES = ["mi", "ci", "vi", "li", "ŝi", "ĝi", "ni", "ili", "oni", "si"]

INICIOS_CORRELATIVOS = ["ki", "ti", "i", "ĉi", "neni"]
FINALES_CORRELATIVOS = {"u": "PRON", "o": "PRON", "a": "DET", "es": "DET",
                        "e": "ADV", "am": "ADV", "el": "ADV", "al": "ADV", "om": "ADV"}

PREPOSICIONES = {"al", "anstataŭ", "antaŭ", "apud", "ĉe", "ĉirkaŭ", "da", "de", "dum", "ekde",
                 "ekster", "el", "en", "ĝis", "inter", "je", "kontraŭ", "krom", "kun", "laŭ",
                 "malantaŭ", "malgraŭ", "per", "po", "por", "post", "preter", "pri", "pro",
                 "sen", "sub", "super", "sur", "tra", "trans"}
CONJUNCIONES = {"kaj", "aŭ", "sed", "nek"}
CONJUNCIONES_SUB = {"ke", "se", "ĉar", "kvankam", "kvazaŭ", "ol"}
PARTICULAS = {"ne", "ja", "jen", "ajn", "ĉi", "ĉu", "ju", "des", "mem"}
ADVERBIOS = {"ankaŭ", "ankoraŭ", "almenaŭ", "apenaŭ", "baldaŭ", "hieraŭ", "hodiaŭ", "morgaŭ",
             "preskaŭ", "nun", "jam", "tre", "tro", "tuj", "for", "plu", "pli", "plej", "nur",
             "eĉ", "ĵus", "do", "tamen", "ambaŭ", "jes"}
INTERJECCIONES = {"adiaŭ", "ha", "he", "ho", "hu", "nu", "ve", "fi", "hura"}
NUMEROS = {"nul", "unu", "du", "tri", "kvar", "kvin", "ses", "sep", "ok", "naŭ", "dek", "cent", "mil"}
AUXILIARES = {"esti", "povi", "devi", "voli"}


def crear_lexico():
    lexico = {}   # palabra -> (pos, lema)

    def añadir(palabra, pos, lema):
        lexico.setdefault(palabra, (pos, lema))

    añadir("la", "DET", "la")
    añadir("l'", "DET", "la")

    for p in PRONOMBRES:
        añadir(p, "PRON", p)
        añadir(p + "n", "PRON", p)                 # acusativo: lin
        for fin in ["a", "aj", "an", "ajn"]:       # posesivos: lia, liaj, lian, liajn
            añadir(p + fin, "DET", p + "a")

    for inicio in INICIOS_CORRELATIVOS:
        for final, pos in FINALES_CORRELATIVOS.items():
            base = inicio + final
            añadir(base, pos, base)
            if final in ("u", "a"):                # kiuj, kiun, kiujn
                for extra in ["j", "n", "jn"]:
                    añadir(base + extra, pos, base)
            elif final in ("o", "e"):              # kion, kien
                añadir(base + "n", pos, base)

    for grupo, pos in [(PREPOSICIONES, "ADP"), (CONJUNCIONES, "CCONJ"),
                       (CONJUNCIONES_SUB, "SCONJ"), (PARTICULAS, "PART"),
                       (ADVERBIOS, "ADV"), (INTERJECCIONES, "INTJ"), (NUMEROS, "NUM")]:
        for palabra in grupo:
            añadir(palabra, pos, palabra)
    return lexico


LEXICO = crear_lexico()
CORRELATIVOS_U = {i + "u" for i in INICIOS_CORRELATIVOS}   # kiu, tiu, iu, ĉiu, neniu
print("Palabras en el léxico:", len(LEXICO))

## 2.2 Stop-words

Partimos de la lista `vortoj.json` del repositorio guía, que tenemos guardada en
`resources/vortoj.json`, y nos quedamos con las palabras marcadas como `"ignorinda": true`.
Se lee del disco; si el cuaderno se ejecuta fuera del repositorio (por ejemplo en Colab), se lee ese
mismo fichero de nuestro GitHub. Para que el cuaderno sobreviva a un fallo de GitHub, como pide el
enunciado, si tampoco se puede leer de ahí se usa una copia de esas palabras escrita en el código
(`COPIA_STOPWORDS`).

Le añadimos lo que le falta: los correlativos (`kiu`, `tio`, `ĉiuj`...), todas las palabras de
nuestro léxico y los verbos auxiliares. Además comprobamos tanto la palabra como su lema, así
`estas` y `estis` cuentan como stop-words porque su lema es `esti`.

In [ ]:
# Copia de las palabras con "ignorinda": true de resources/vortoj.json. Solo se usa si no se puede
# leer el fichero ni del disco ni de GitHub
COPIA_STOPWORDS = [
    "adiaŭ", "ajn", "al", "ankaŭ", "ankoraŭ", "anstataŭ", "antaŭ", "apenaŭ", "apud", "aŭ",
    "baldaŭ", "bonvoli", "cent", "ci", "da", "de", "dek", "des", "do", "doktoro", "du", "dum",
    "eble", "ekster", "ekzemple", "el", "en", "esti", "eĉ", "for", "fraŭlo", "ha", "havi", "he",
    "hieraŭ", "ho", "hodiaŭ", "hu", "ili", "inter", "ja", "jam", "je", "jen", "jes", "ju", "kaj",
    "ke", "kontraŭ", "krom", "kun", "kvankam", "kvar", "kvazaŭ", "kvin", "la", "laŭ", "li",
    "male", "malgraŭ", "mem", "mi", "mil", "minus", "morgaŭ", "naŭ", "ne", "nek", "nepre", "ni",
    "nu", "nul", "nun", "nur", "ok", "ol", "oni", "per", "plej", "pli", "plu", "plus", "po",
    "por", "post", "povi", "preskaŭ", "preter", "pri", "pro", "se", "sed", "sen", "sep", "ses",
    "si", "sinjoro", "sub", "super", "supre", "sur", "tamen", "tra", "trans", "tre", "tri", "tro",
    "tuj", "tute", "unu", "ve", "verŝajne", "vi", "ĉar", "ĉe", "ĉi", "ĉirkaŭ", "ĉu", "ĝi", "ĝis",
    "ĵus", "ŝi"
]


def leer_vortoj():
    # Devuelve el contenido de vortoj.json: 1º del disco, 2º de nuestro GitHub, 3º la copia de arriba
    if os.path.exists(RUTA_STOPWORDS):
        with open(RUTA_STOPWORDS, encoding="utf-8") as f:
            print("Stop-words leídas de", RUTA_STOPWORDS)
            return json.load(f)
    try:
        respuesta = requests.get(URL_STOPWORDS, timeout=10)
        respuesta.raise_for_status()
        datos = respuesta.json()
        print("Stop-words leídas de", URL_STOPWORDS)
        return datos
    except Exception as e:
        print(f"No se pudo leer {URL_STOPWORDS} ({type(e).__name__}): se usa COPIA_STOPWORDS")
        return {"vortoj": [{"vorto": v, "ignorinda": True} for v in COPIA_STOPWORDS]}


def cargar_stopwords():
    datos = leer_vortoj()
    stopwords = {e["vorto"] for e in datos["vortoj"] if e.get("ignorinda")}

    stopwords |= {lema for pos, lema in LEXICO.values()}   # correlativos, preposiciones...
    stopwords |= AUXILIARES
    return stopwords


STOP_WORDS = cargar_stopwords()
print("Total de stop-words:", len(STOP_WORDS))

## 2.3 Clases `Token` y `Doc`

Igual que en spaCy: un `Doc` es la lista de `Token`s de un texto. Se puede recorrer con un `for`,
acceder por posición (`doc[0]`) y sacar un trozo (`doc[1:4]`).

In [ ]:
class Token:
    def __init__(self, text, i):
        self.text = text      # texto original
        self.i = i            # posición dentro del Doc
        self.lemma_ = ""
        self.pos_ = ""
        self.morph = {}       # rasgos morfológicos: número, caso, tiempo...
        self.is_stop = False

    @property
    def is_punct(self):
        return self.pos_ == "PUNCT"

    @property
    def is_alpha(self):
        return self.text.isalpha()

    def __repr__(self):
        return self.text


class Doc:
    def __init__(self, text, tokens):
        self.text = text
        self.tokens = tokens

    def __iter__(self):
        return iter(self.tokens)

    def __len__(self):
        return len(self.tokens)

    def __getitem__(self, i):
        return self.tokens[i]

    def tabla(self):
        # Análisis de cada token en forma de tabla
        return pd.DataFrame([{"text": t.text, "lemma": t.lemma_, "pos": t.pos_,
                              "morph": t.morph, "is_stop": t.is_stop} for t in self.tokens])

    def __repr__(self):
        return self.text

## 2.4 Tokenizador

Una expresión regular que reconoce, en este orden: abreviaturas comunes (`km/h`, `ekz.`, `ktp.`,
`k.t.p.`, `t.e.`), URLs, números (`1950`, `3,5` y también
ordinales como `20-a`), palabras (con letras del esperanto `ĉ ĝ ĥ ĵ ŝ ŭ` y compuestos con guion
como `Dizel-lokomotivo`) y cualquier signo de puntuación suelto.

Las abreviaturas van primero para que no se partan: si `ekz.` (*ekzemple*, 'por ejemplo') se
separara en `ekz` + `.`, ese punto parecería un final de frase y la palabra siguiente dejaría de
reconocerse como nombre propio (`Mi vidis ekz. Parizon`).

In [ ]:
PATRON_TOKENS = re.compile(r'''
      (?<!\w)(?i:km/h|ekz\.|ktp\.|k\.t\.p\.|t\.e\.)(?!\w)   # abreviaturas: km/h, ekz., ktp., t.e.
    | https?://\S+ | www\.\S+              # URLs
    | \d+(?:[.,]\d+)*(?:-[a-zĉĝĥĵŝŭ]+)?     # números: 1950, 3,5, 20-a
    | [lL]'(?=\s)                          # artículo abreviado: l'
    | [^\W\d_][^\W_]*(?:-[^\W_]+)*          # palabras, también con guion
    | [^\w\s]                              # signos de puntuación
''', re.VERBOSE)


def tokenizar(texto):
    return PATRON_TOKENS.findall(texto)


tokenizar("La Dizel-lokomotivo veturis je 120 km/h en la 20-a jarcento.")

## 2.5 Análisis de cada palabra: POS, lema y morfología

`analizar_palabra()` mira una palabra (en minúsculas) y devuelve su categoría, su lema y sus
rasgos. Las reglas se prueban en orden:

1. Si está en el **léxico**, se usa lo que dice el léxico.
2. **Números**: en cifras (`1950`), ordinales en cifras (`20-a`) y compuestos (`dudek` = 20).
3. **Palabras extranjeras**: si tiene alguna letra que no existe en esperanto (`q`, `w`, `x`, `y`,
   `é`, `ó`...), es una palabra de otro idioma (`X`, con el rasgo `{"extranjera": True}`).
4. **Participios**: `konstruita` → lema `konstrui`. Hay palabras que parecen participios pero no lo
   son (`konstanta`, `brita`, `privata`...); las fuimos encontrando al revisar el corpus y las
   guardamos en una lista de excepciones. Antes de mirar la lista se quita lo que va antes de un
   guion y los prefijos más comunes, para que `senpilota` (`sen` + `pilota`) y `post-milita` no se
   tomen por participios.
5. **Verbos**: `-as`, `-is`, `-os`, `-us`, `-u`, `-i` → lema en `-i`.
6. **Sustantivos**, **adjetivos** y **adverbios** por su terminación.
7. Si nada encaja, `X` (palabra **no reconocida**). Aquí caen, por ejemplo, unidades y abreviaturas
   como `kg`, `t` o `cm`: no son esperanto, pero tampoco son de otro idioma.

**Lematizar** en esperanto es cambiar la terminación por la de diccionario: `trajnojn` → `trajno`,
`rapidaj` → `rapida`, `veturis` → `veturi`.

Usamos `lru_cache` para no analizar dos veces la misma palabra, porque en el corpus se repiten
mucho.

In [ ]:
ALFABETO = set("abcĉdefgĝhĥijĵklmnoprsŝtuŭvz")

# Raíces que parecen participios pero no lo son (konst-anta, br-ita, priv-ata...)
NO_PARTICIPIOS = {
    "konst", "gig", "eleg", "elef", "horiz", "aŭtom", "priv", "apar", "klim", "sold", "kandid",
    "rezult", "sen", "kval", "pil", "rob", "patri", "idi", "diam", "mil", "parazi", "str",
    "ril", "jezui", "fosf", "merk", "adiab", "bo", "profi", "sport", "kroko", "demokr",
    "ŝt", "fr", "br", "gr", "pl", "sp", "tr", "kv", "fl", "oh",
}

# Prefijos que se quitan antes de mirar NO_PARTICIPIOS: sen-pil-ota -> pil, ne-konst-anta -> konst
PREFIJOS = ("sen", "sur", "post", "ekster", "mal", "ne", "antaŭ", "inter", "sub", "super", "tra",
            "trans", "pri", "kontraŭ", "kun", "el", "en")


def es_falso_participio(raiz):
    # Con guion solo cuenta la última parte: post-mil(ita) -> mil
    raiz = raiz.split("-")[-1]
    candidatas = {raiz} | {raiz[len(p):] for p in PREFIJOS if raiz.startswith(p)}
    return any(r in NO_PARTICIPIOS for r in candidatas)


TIEMPOS = {"as": "presente", "is": "pasado", "os": "futuro", "us": "condicional",
           "u": "imperativo", "i": "infinitivo"}
PARTICIPIOS = {"ant": "activo presente", "int": "activo pasado", "ont": "activo futuro",
               "at": "pasivo presente", "it": "pasivo pasado", "ot": "pasivo futuro"}


def numero_y_caso(final):
    # A partir de -j y -n: trajnojn -> plural, acusativo
    return {"numero": "plural" if "j" in final else "singular",
            "caso": "acusativo" if final.endswith("n") else "nominativo"}


@lru_cache(maxsize=None)
def analizar_palabra(palabra):
    # 1. Léxico
    if palabra in LEXICO:
        pos, lema = LEXICO[palabra]
        return pos, lema, {}

    # 2. Números
    if re.fullmatch(r"\d+(?:[.,]\d+)*", palabra):
        return "NUM", palabra, {}
    if re.fullmatch(r"\d+-aj?n?", palabra):                    # 20-a, 1950-aj
        return "ADJ", palabra, {}
    if re.fullmatch(r"(?:(?:du|tri|kvar|kvin|ses|sep|ok|naŭ)?(?:dek|cent|mil))+", palabra):
        return "NUM", palabra, {}                                 # dudek, tricent...

    # 3. Palabras extranjeras (letras que no existen en esperanto). Se marcan con un rasgo para
    #    distinguirlas de las no reconocidas (regla 7), como las unidades kg, t o cm
    if any(c.isalpha() and c not in ALFABETO for c in palabra):
        return "X", palabra, {"extranjera": True}

    # 4. Participios adjetivales y adverbiales
    m = re.fullmatch(r"(.{2,}?)(ant|int|ont|at|it|ot)(a|aj|an|ajn|e)", palabra)
    if m and not es_falso_participio(m.group(1)):
        raiz, sufijo, final = m.groups()
        morph = {"participio": PARTICIPIOS[sufijo]}
        if final == "e":
            return "ADV", raiz + "i", morph
        return "ADJ", raiz + "i", {**morph, **numero_y_caso(final)}

    # 5. Verbos
    m = re.fullmatch(r"(.{2,}?)(as|is|os|us|u|i)", palabra)
    if m:
        raiz, final = m.groups()
        pos = "AUX" if raiz + "i" in AUXILIARES else "VERB"
        return pos, raiz + "i", {"tiempo": TIEMPOS[final]}

    # 6. Sustantivos, adjetivos y adverbios
    m = re.fullmatch(r"(.+?)(o|oj|on|ojn)", palabra)
    if m:
        return "NOUN", m.group(1) + "o", numero_y_caso(m.group(2))
    m = re.fullmatch(r"(.+?)(a|aj|an|ajn)", palabra)
    if m:
        return "ADJ", m.group(1) + "a", numero_y_caso(m.group(2))
    m = re.fullmatch(r"(.+?)(e|en)", palabra)
    if m:
        return "ADV", m.group(1) + "e", {}                        # hejmen -> hejme
    if palabra.endswith("aŭ") and len(palabra) > 3:
        return "ADV", palabra, {}

    # 7. No reconocida
    return "X", palabra, {}


for p in ["trajnojn", "rapidaj", "veturis", "konstruita", "konstanta", "kiujn", "dudek", "aviación"]:
    print(f"{p:<12}", analizar_palabra(p))

## 2.6 Reglas de contexto

Algunas cosas no se pueden saber mirando la palabra sola, así que después del análisis aplicamos
unas pocas reglas que miran a los vecinos:

* **Nombres propios**: una palabra en mayúscula que no está al principio de la frase (`en Francio`,
  `de Airbus`) o que es una sigla (`TGV`) es `PROPN`. Su lema conserva la mayúscula y pierde el
  acusativo: `Francion` → `Francio`, `Parizon` → `Parizo`. Hay que tener cuidado, porque muchos
  nombres extranjeros también acaban en `-on` (`London`, `Washington`, `Stephenson`). Por eso no
  quitamos la `-n` si la palabra tiene letras que no existen en esperanto o acaba en `-son`,
  `-ton`, `-don` o `-gon`. No es perfecto: `Konkordon` se queda igual y `Elon` pasa a `Elo`.

  Al principio de frase no se puede usar la mayúscula, y algunos nombres parecen verbos
  conjugados: `Airbus` se analizaría como el verbo `airbi` en condicional. Lo tratamos como
  nombre propio si le sigue otro verbo conjugado (`Airbus konstruas...`), porque en esperanto casi
  nunca van dos seguidos, o si está en una lista corta de nombres frecuentes del corpus (`Airbus`,
  `Louis`, `Francis`, `Lucas`). Que le siga un número o un nombre no sirve como pista, porque pasa
  también con verbos de verdad: `Mortis 583 homoj`, `Venkis Georges...`.
* **Correlativos en `-u`** (`tiu`, `kiu`...): son determinante si van delante de un sustantivo o
  adjetivo (`tiu trajno`) y pronombre si no (`kiu venis?`).
* **`povi`, `devi`, `voli`**: son auxiliares solo si les sigue un infinitivo (`ni povas veturi`);
  si no, son verbos normales (`ni volas pacon`).

In [ ]:
def es_inicio_de_frase(doc, i):
    return i == 0 or doc[i - 1].text in {".", "!", "?", ":"}


# Nombres propios frecuentes del corpus que parecen verbos conjugados (Airbus -> 'airbi')
NOMBRES_COMO_VERBOS = {"airbus", "louis", "francis", "lucas"}


def es_verbo_conjugado(t):
    return t.pos_ in ("VERB", "AUX") and t.morph.get("tiempo") not in (None, "infinitivo")


def etiquetar(doc):
    # Primero cada palabra por separado
    for t in doc:
        if not any(c.isalnum() for c in t.text):
            t.pos_, t.lemma_ = "PUNCT", t.text
            continue
        if t.text.startswith(("http", "www")):
            t.pos_, t.lemma_ = "X", t.text
            continue
        t.pos_, t.lemma_, t.morph = analizar_palabra(t.text.lower())

    # Después, las reglas de contexto
    for i, t in enumerate(doc):
        siguiente = doc[i + 1] if i + 1 < len(doc) else None

        # Nombres propios
        if t.text[0].isupper() and t.text.lower() not in LEXICO:
            sigla = t.text.isupper() and len(t.text) > 1
            # Al principio de frase, un "verbo" en mayúscula es nombre propio si está en la lista
            # o si le sigue otro verbo conjugado (Airbus konstruas...): casi nunca van dos seguidos
            falso_verbo = es_verbo_conjugado(t) and (
                t.text.lower() in NOMBRES_COMO_VERBOS
                or (siguiente is not None and es_verbo_conjugado(siguiente)))
            if sigla or t.pos_ == "X" or not es_inicio_de_frase(doc, i) or falso_verbo:
                t.pos_ = "PROPN"
                t.lemma_ = re.sub(r"(io|ujo)n$", r"\1", t.text)    # Francion -> Francio
                t.lemma_ = re.sub(r"ojn?$", "o", t.lemma_)         # Francioj -> Francio
                # Parizon -> Parizo. Muchos nombres extranjeros también acaban en -on (London,
                # Washington, Stephenson): no se tocan si tienen letras que no existen en
                # esperanto o si acaban en -son, -ton, -don o -gon
                if all(c in ALFABETO for c in t.lemma_.lower() if c.isalpha()):
                    t.lemma_ = re.sub(r"(?<![stdg])on$", "o", t.lemma_)
                t.morph = {}

        # Correlativos en -u: determinante o pronombre
        if t.lemma_ in CORRELATIVOS_U:
            t.pos_ = "DET" if siguiente and siguiente.pos_ in ("NOUN", "ADJ", "PROPN") else "PRON"

        # povi / devi / voli: auxiliar solo si va seguido de infinitivo
        if t.pos_ == "AUX" and t.lemma_ != "esti":
            siguientes = [u for u in doc.tokens[i + 1:i + 4] if u.pos_ not in ("ADV", "PART")]
            if not (siguientes and siguientes[0].morph.get("tiempo") == "infinitivo"):
                t.pos_ = "VERB"
    return doc


def marcar_stopwords(doc):
    for t in doc:
        t.is_stop = t.text.lower() in STOP_WORDS or t.lemma_.lower() in STOP_WORDS
    return doc

## 2.7 La clase `Esperanto` (el `nlp` de spaCy)

Junta todo: tokeniza el texto, crea el `Doc` y le pasa los componentes del pipeline en orden.

In [ ]:
class Esperanto:
    def __init__(self):
        self.pipeline = [("tagger", etiquetar), ("stopwords", marcar_stopwords)]

    @property
    def pipe_names(self):
        return [nombre for nombre, _ in self.pipeline]

    def __call__(self, texto):
        palabras = tokenizar(texto)
        doc = Doc(texto, [Token(p, i) for i, p in enumerate(palabras)])
        for _, componente in self.pipeline:
            doc = componente(doc)
        return doc

    def pipe(self, textos):
        # Para procesar muchos textos seguidos, como nlp.pipe() de spaCy
        for texto in textos:
            yield self(texto)


nlp = Esperanto()
nlp.pipe_names

### Ejemplo de uso

In [ ]:
doc = nlp("La rapidaj elektraj trajnoj veturis inter Parizo kaj Liono, kaj la "
          "konstruita flughaveno povos akcepti multajn aviadilojn.")
doc.tabla()

In [ ]:
for token in nlp("Tiu aŭto, kiun mi aĉetis, estas rapida. Ĉu vi vidis ĝin?"):
    print(f"{token.text:<8} {token.lemma_:<8} {token.pos_:<6} stop={token.is_stop}")

## 2.8 Evaluación del modelo

### Pruebas de casos concretos
Cada línea comprueba una regla: el texto, la posición del token, el lema y la categoría que
esperamos. Si alguna falla, la celda se detiene con un error (`assert`), para que no pase
desapercibido al ejecutar el cuaderno entero.

In [ ]:
pruebas = [
    # texto, posición, lema, pos
    ("Mi vidas rapidajn trajnojn.", 2, "rapida", "ADJ"),
    ("Mi vidas rapidajn trajnojn.", 3, "trajno", "NOUN"),
    ("La aviadilo flugis.", 2, "flugi", "VERB"),
    ("La aviadilo flugos.", 2, "flugi", "VERB"),
    ("La konstruita stacio", 1, "konstrui", "ADJ"),
    ("La veturantoj", 1, "veturanto", "NOUN"),
    ("La piloto flugas.", 1, "piloto", "NOUN"),          # -ot- no es participio aquí
    ("La ŝtato kaj la aparato", 1, "ŝtato", "NOUN"),
    ("Ŝi parolis ridante.", 2, "ridi", "ADV"),           # participio adverbial
    ("La konstanta rapido", 1, "konstanta", "ADJ"),
    ("La brita fervojo", 1, "brita", "ADJ"),
    ("La senpilota aviadilo", 1, "senpilota", "ADJ"),
    ("post-milita tempo", 0, "post-milita", "ADJ"),
    ("La uzata aviadilo", 1, "uzi", "ADJ"),
    ("Ĝi estas rapida.", 1, "esti", "AUX"),
    ("Ni povas veturi.", 1, "povi", "AUX"),
    ("Ni volas pacon.", 1, "voli", "VERB"),
    ("Mi vidis lin.", 2, "li", "PRON"),
    ("Iliajn aŭtojn", 0, "ilia", "DET"),
    ("Tiujn aŭtojn", 0, "tiu", "DET"),
    ("Kiu venis?", 0, "kiu", "PRON"),
    ("Ĝi veturas rapide.", 2, "rapide", "ADV"),
    ("Ili iris hejmen.", 2, "hejme", "ADV"),
    ("En 1950 estis du trajnoj.", 1, "1950", "NUM"),
    ("La dudek aviadiloj", 1, "dudek", "NUM"),
    ("Ĝi flugis al Francion.", 3, "Francio", "PROPN"),
    ("Ĝi flugis al Parizon.", 3, "Parizo", "PROPN"),
    ("Ni vizitis Londonon.", 2, "Londono", "PROPN"),
    ("La TGV estas rapida.", 1, "TGV", "PROPN"),
    ("Airbus konstruas aviadilojn.", 0, "Airbus", "PROPN"),
    ("Airbus A380 kaj Boeing 747", 0, "Airbus", "PROPN"),
    ("Flugu!", 0, "flugi", "VERB"),
    ("Mortis 583 homoj.", 0, "morti", "VERB"),
    ("aviones y trenes", 0, "aviones", "X"),
    ("En London", 1, "London", "PROPN"),
    ("Mi vidis ekz. Parizon", 3, "Parizo", "PROPN")
]

errores = 0
for texto, i, lema, pos in pruebas:
    t = nlp(texto)[i]
    if (t.lemma_, t.pos_) != (lema, pos):
        errores += 1
        print(f"FALLO en '{texto}': {t.text} -> {t.lemma_}/{t.pos_}, esperado {lema}/{pos}")

print(f"{len(pruebas) - errores} de {len(pruebas)} pruebas correctas")
assert errores == 0, f"{errores} pruebas han fallado"

# morfología y stop-words
assert nlp("Mi vidas trajnojn.")[2].morph == {"numero": "plural", "caso": "acusativo"}
assert nlp("Ĝi flugis.")[1].morph == {"tiempo": "pasado"}
assert [t.is_stop for t in nlp("Estis tiuj trajnoj")] == [True, True, False]

# abreviaturas: no se parten
assert tokenizar("je 120 km/h, ekz. trajnoj, aŭtoj k.t.p.") == \
    ["je", "120", "km/h", ",", "ekz.", "trajnoj", ",", "aŭtoj", "k.t.p."]

### Precisión sobre frases etiquetadas a mano

Etiquetamos a mano dos grupos de frases con el formato `palabra/POS/lema`:

1. **Frases nuestras** (10 frases), que usamos mientras desarrollábamos las reglas. El resultado
   es optimista porque las reglas se ajustaron con ellas.
2. **Frases reales del dataset** (7 frases elegidas al azar), etiquetadas después de terminar las
   reglas y sin volver a tocarlas. Esta es la medida más realista.

In [ ]:
FRASES_PROPIAS = '''
La/DET/la unua/ADJ/unua fervojo/NOUN/fervojo por/ADP/por pasaĝeroj/NOUN/pasaĝero estis/AUX/esti malfermita/ADJ/malfermi en/ADP/en Anglio/PROPN/Anglio en/ADP/en 1825/NUM/1825 ./PUNCT/.
Modernaj/ADJ/moderna rapidtrajnoj/NOUN/rapidtrajno povas/AUX/povi veturi/VERB/veturi pli/ADV/pli rapide/ADV/rapide ol/SCONJ/ol aŭtomobiloj/NOUN/aŭtomobilo ./PUNCT/.
La/DET/la piloto/NOUN/piloto kondukis/VERB/konduki la/DET/la aviadilon/NOUN/aviadilo al/ADP/al la/DET/la flughaveno/NOUN/flughaveno de/ADP/de Madrido/PROPN/Madrido ./PUNCT/.
Multaj/ADJ/multa homoj/NOUN/homo preferas/VERB/preferi vojaĝi/VERB/vojaĝi per/ADP/per aŭtobuso/NOUN/aŭtobuso ĉar/SCONJ/ĉar ĝi/PRON/ĝi estas/AUX/esti malmultekosta/ADJ/malmultekosta ./PUNCT/.
Kiam/ADV/kiam la/DET/la motoro/NOUN/motoro ekfunkciis/VERB/ekfunkcii ,/PUNCT/, la/DET/la ŝoforo/NOUN/ŝoforo rimarkis/VERB/rimarki ke/SCONJ/ke la/DET/la bremsoj/NOUN/bremso ne/PART/ne funkcias/VERB/funkcii ./PUNCT/.
Elektraj/ADJ/elektra lokomotivoj/NOUN/lokomotivo ricevas/VERB/ricevi energion/NOUN/energio el/ADP/el supra/ADJ/supra kontaktdrato/NOUN/kontaktdrato ./PUNCT/.
Tiuj/DET/tiu helikopteroj/NOUN/helikoptero flugos/VERB/flugi super/ADP/super la/DET/la montoj/NOUN/monto dum/ADP/dum tri/NUM/tri horoj/NOUN/horo ./PUNCT/.
La/DET/la flugkompanio/NOUN/flugkompanio aĉetis/VERB/aĉeti dudek/NUM/dudek novajn/ADJ/nova aviadilojn/NOUN/aviadilo de/ADP/de Airbus/PROPN/Airbus ./PUNCT/.
Ŝi/PRON/ŝi parkumis/VERB/parkumi sian/DET/sia aŭton/NOUN/aŭto apud/ADP/apud la/DET/la stacidomo/NOUN/stacidomo kaj/CCONJ/kaj iris/VERB/iri hejmen/ADV/hejme ./PUNCT/.
La/DET/la vagonoj/NOUN/vagono estas/AUX/esti tirataj/ADJ/tiri de/ADP/de dizela/ADJ/dizela lokomotivo/NOUN/lokomotivo ,/PUNCT/, kiu/PRON/kiu atingas/VERB/atingi 120/NUM/120 kilometrojn/NOUN/kilometro hore/ADV/hore ./PUNCT/.
'''

FRASES_REALES = '''
Ĝi/PRON/ĝi povas/AUX/povi esti/AUX/esti influita/ADJ/influi de/ADP/de potencotrajna/ADJ/potencotrajna dezajno/NOUN/dezajno ,/PUNCT/, lubrikaĵa/ADJ/lubrikaĵa speco/NOUN/speco kaj/CCONJ/kaj temperaturo/NOUN/temperaturo kaj/CCONJ/kaj multaj/ADJ/multa aliaj/ADJ/alia faktoroj/NOUN/faktoro ./PUNCT/.
Aviadiloj/NOUN/aviadilo povas/AUX/povi anstataŭe/ADV/anstataŭe uzi/VERB/uzi jetmotorojn/NOUN/jetmotoro kaj/CCONJ/kaj helikopteroj/NOUN/helikoptero povas/AUX/povi anstataŭe/ADV/anstataŭe uzi/VERB/uzi turboŝaftojn/NOUN/turboŝafto ,/PUNCT/, kiuj/PRON/kiu ambaŭ/PRON/ambaŭ estas/AUX/esti specoj/NOUN/speco de/ADP/de turbinoj/NOUN/turbino ./PUNCT/.
Oni/PRON/oni povas/AUX/povi la/DET/la sekurecon/NOUN/sekureco de/ADP/de la/DET/la aŭtomobiloj/NOUN/aŭtomobilo plibonigi/VERB/plibonigi per/ADP/per administraj/ADJ/administra kaj/CCONJ/kaj konstruaj/ADJ/konstrua rimedoj/NOUN/rimedo ./PUNCT/.
Harmon/PROPN/Harmon esprimis/VERB/esprimi favoron/NOUN/favoro al/ADP/al Esperanto/PROPN/Esperanto en/ADP/en la/DET/la franca/ADJ/franca Esperanto-revuo/NOUN/esperanto-revuo La/DET/la Movado/PROPN/Movado ./PUNCT/.
Selden/PROPN/Selden postulis/VERB/postuli patenton/NOUN/patento por/ADP/por aŭto/NOUN/aŭto en/ADP/en 1879/NUM/1879 ,/PUNCT/, sed/CCONJ/sed la/DET/la patentopostulo/NOUN/patentopostulo elvalidiĝis/VERB/elvalidiĝi ĉar/SCONJ/ĉar la/DET/la vehiklo/NOUN/vehiklo neniam/ADV/neniam estis/AUX/esti konstruita/ADJ/konstrui ./PUNCT/.
Tion/PRON/tio oni/PRON/oni provas/VERB/provi obstakli/VERB/obstakli per/ADP/per rentgenaj/ADJ/rentgena aparatoj/NOUN/aparato kaj/CCONJ/kaj hundoj/NOUN/hundo ĉe/ADP/ĉe la/DET/la limo/NOUN/limo aŭ/CCONJ/aŭ eĉ/ADV/eĉ landinterne/ADV/landinterne ./PUNCT/.
La/DET/la kapa/ADJ/kapa salono/NOUN/salono havas/VERB/havi 110/NUM/110 sidajn/ADJ/sida kaj/CCONJ/kaj 140/NUM/140 starajn/ADJ/stara lokojn/NOUN/loko ,/PUNCT/, la/DET/la dua/ADJ/dua salono/NOUN/salono –/PUNCT/– 60/NUM/60 sidajn/ADJ/sida kaj/CCONJ/kaj 40/NUM/40 starajn/ADJ/stara ./PUNCT/.
'''


def evaluar(frases_anotadas):
    total = pos_bien = lema_bien = 0
    fallos = []
    for linea in frases_anotadas.strip().splitlines():
        anotacion = [palabra.rsplit("/", 2) for palabra in linea.split()]
        doc = nlp(" ".join(palabra for palabra, _, _ in anotacion))
        for token, (palabra, pos, lema) in zip(doc, anotacion):
            total += 1
            pos_bien += token.pos_ == pos
            lema_bien += token.lemma_ == lema
            if token.pos_ != pos or token.lemma_ != lema:
                fallos.append((palabra, f"{token.pos_}/{token.lemma_}", f"{pos}/{lema}"))
    return total, pos_bien / total, lema_bien / total, fallos


resultados = []
for nombre, frases in [("Frases propias", FRASES_PROPIAS), ("Frases reales del dataset", FRASES_REALES)]:
    total, acierto_pos, acierto_lema, fallos = evaluar(frases)
    resultados.append({"conjunto": nombre, "tokens": total,
                       "acierto POS": f"{acierto_pos:.1%}", "acierto lema": f"{acierto_lema:.1%}"})
    for palabra, predicho, correcto in fallos:
        print(f"[{nombre}] {palabra}: predicho {predicho}, correcto {correcto}")

pd.DataFrame(resultados)

**Errores.** Los fallos en las frases reales vienen de casos que no se resuelven solo con reglas
de terminación:

* Nombres propios al principio de la frase (`Harmon`, `Selden`): como la primera palabra siempre
  va en mayúscula, no los distinguimos de una palabra normal. Además `Selden` acaba en `-en` y
  parece un adverbio.
* Palabras con mayúscula de título (`Esperanto-revuo`) que tomamos por nombre propio.
* `povas` separado de su infinitivo por varias palabras, o `ambaŭ` usado como pronombre.

Para clasificar textos por tema estos errores importan poco, porque lo que más pesa son los lemas
de sustantivos, verbos y adjetivos.

## 2.9 El modelo aplicado al dataset

In [ ]:
docs = list(nlp.pipe(df["text"]))

conteo_pos = Counter(t.pos_ for d in docs for t in d)
palabras = sum(n for pos, n in conteo_pos.items() if pos != "PUNCT")
print("Tokens:", sum(conteo_pos.values()))
print(f"Palabras no reconocidas (X): {conteo_pos['X'] / palabras:.1%}")
print(f"Stop-words: {sum(t.is_stop for d in docs for t in d) / palabras:.1%}")

etiquetas, cantidades = zip(*conteo_pos.most_common())
plt.figure(figsize=(9, 4))
plt.bar(etiquetas, cantidades)
plt.title("Categorías gramaticales en el dataset")
plt.ylabel("número de tokens")
plt.show()

### Medida del ruido: palabras extranjeras

Aunque ya filtramos los párrafos en otros idiomas, algunos todavía mezclan esperanto con nombres
o títulos en otras lenguas. Con el modelo podemos contar en cada párrafo qué proporción de
palabras son **extranjeras**, es decir, las que tienen letras que no existen en esperanto (rasgo
`extranjera` de la regla 3).

No contamos las palabras **no reconocidas** (regla 7), aunque también sean `X`. En una primera
versión las contábamos y el filtro descartaba párrafos válidos de `fervojo` con datos técnicos de
locomotoras, llenos de unidades (`13 t vaporo : 7 kg/PSih donas ĉirkaŭ 1857 PS`). Eso no es texto
en otro idioma.

Tras el filtrado de la Fase 1 (regla «otro idioma») ya no queda ningún párrafo con más de un 20 %
de palabras extranjeras, así que este filtro no descarta nada. Lo dejamos como comprobación: si se
regenera el dataset y se cuela algún párrafo en otro idioma, aquí se vería y se quitaría.

In [ ]:
def proporcion_extranjeras(doc):
    # Solo cuentan las palabras con letras que no existen en esperanto (regla 3), no las unidades
    # ni abreviaturas no reconocidas (kg, t, cm...)
    palabras = [t for t in doc if t.is_alpha]
    return sum(t.morph.get("extranjera", False) for t in palabras) / max(len(palabras), 1)


df["extranjeras"] = [proporcion_extranjeras(d) for d in docs]
ruidosos = df[df["extranjeras"] > 0.2]
print(f"Párrafos con más de un 20% de palabras extranjeras: {len(ruidosos)} de {len(df)}")
mantener = (df["extranjeras"] <= 0.2).to_numpy()
df = df[mantener].reset_index(drop=True)
docs = [d for d, m in zip(docs, mantener) if m]
print("Párrafos tras descartar >20% de palabras extranjeras:", len(df))
if len(ruidosos) > 0:
    display(ruidosos.head(8))

### Lemas más frecuentes por clase

Si el modelo funciona bien, los lemas más frecuentes de cada clase (quitando stop-words) deberían
ser claramente de ese tema.

In [ ]:
CATEGORIAS_CON_SIGNIFICADO = {"NOUN", "ADJ", "VERB", "PROPN", "ADV"}

fig, ejes = plt.subplots(1, 3, figsize=(16, 4.5))
for eje, clase in zip(ejes, sorted(df["class"].unique())):
    conteo = Counter(t.lemma_ for d, c in zip(docs, df["class"]) if c == clase for t in d
                     if t.pos_ in CATEGORIAS_CON_SIGNIFICADO and not t.is_stop and len(t.lemma_) > 2)
    lemas, n = zip(*conteo.most_common(15))
    eje.barh(lemas[::-1], n[::-1])
    eje.set_title(clase)
plt.suptitle("15 lemas más frecuentes por clase")
plt.tight_layout()
plt.show()

### Cuánto reduce el vocabulario la lematización

Con la lematización `trajno`, `trajnoj`, `trajnon` y `trajnojn` pasan a ser una sola palabra. Eso
hace que la matriz de la Fase 3 tenga menos columnas.

In [ ]:
formas = {t.text.lower() for d in docs for t in d if t.is_alpha}
lemas = {t.lemma_.lower() for d in docs for t in d if t.is_alpha}
print("Palabras distintas sin lematizar:", len(formas))
print("Lemas distintos:", len(lemas))

## 2.10 Preparación para la Fase 3

`lemas_utiles()` convierte un texto en la lista de lemas que nos interesan: sin stop-words,
puntuación, números ni palabras extranjeras. Se la pasamos a `TfidfVectorizer` de scikit-learn con
`analyzer=lemas_utiles`, así toda la tokenización y lematización la hace nuestro modelo.

In [ ]:
def lemas_utiles(texto):
    return [t.lemma_.lower() for t in nlp(texto)
            if not t.is_stop and t.pos_ not in ("PUNCT", "NUM", "X")
            and len(t.lemma_) > 1 and not t.lemma_[0].isdigit()]


print(lemas_utiles(df["text"].iloc[0]))

### Medida del ruido de etiquetado

Cada párrafo hereda la etiqueta del artículo del que sale, y no siempre es correcta: hay párrafos
genéricos o que hablan de otro tema (Fase 1). El enunciado pide reconocer y, si se puede, medir ese
ruido. Para medirlo le pedimos a un clasificador que prediga la clase de cada párrafo y miramos en
cuáles no coincide con la etiqueta.

Usamos **validación cruzada** (`cross_val_predict`, 5 particiones): cada párrafo lo predice un
modelo entrenado con los otros cuatro quintos, que **no lo ha visto**. Si lo predijera un modelo
entrenado con él, casi siempre acertaría y no mediríamos nada.

La celda da dos números:

* **Contradicen su etiqueta con confianza > 0,5**: el modelo da a otra clase más de la mitad de la
  probabilidad. Son los candidatos más claros a estar mal etiquetados.
* **Predicción distinta a la etiqueta (cualquier confianza)**: todos los desacuerdos, también los
  dudosos. Es una cota superior del ruido, porque incluye errores del propio modelo.

Es solo un **análisis exploratorio**: no quitamos estos párrafos ni usamos el resultado para
entrenar o para elegir el modelo, así que no contamina el test de la Fase 3.

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression 
from sklearn.model_selection import StratifiedKFold, cross_val_predict

pipe = make_pipeline(TfidfVectorizer(analyzer=lemas_utiles, min_df=2),
                     LogisticRegression(max_iter=1000, class_weight="balanced"))
proba = cross_val_predict(pipe, df["text"], df["class"],
                          cv=StratifiedKFold(5, shuffle=True, random_state=SEMILLA),
                          method="predict_proba")
clases = np.array(sorted(df["class"].unique()))
df["pred_cv"] = clases[proba.argmax(axis=1)]
df["confianza"] = proba.max(axis=1)

sospechosos = df[(df["pred_cv"] != df["class"]) & (df["confianza"] > 0.5)]
print(f"Contradicen su etiqueta con confianza > 0.5: {len(sospechosos)} de {len(df)}")
print(f"Predicción distinta a la etiqueta (cualquier confianza): {(df['pred_cv'] != df['class']).sum()} de {len(df)}")
sospechosos.sample(min(50, len(sospechosos)), random_state=SEMILLA)[["text", "class", "pred_cv"]]

Solo 5 de 981 párrafos contradicen su etiqueta con claridad, y unos 74 (un 7,5 %) en total, así que
el ruido de etiquetado existe pero es pequeño. En los ejemplos se ven los dos casos de la Fase 1:

* **Párrafos genéricos**, sin vocabulario del tema: el de las tiendas y aseos de las estaciones
  (`vendejoj, manĝejoj, necesejoj...`) no tiene ninguna palabra de tren, y el de los modelos de
  tranvía más fabricados habla de modelos y producción, como los de coches.
* **Párrafos que hablan de otro tema**: el artículo de motores de combustión tiene un párrafo sobre
  los motores de aviones y helicópteros, y el de autobuses habla de un autobús de aeropuerto
  (*Jumbo Jet Coach*). Sus etiquetas son correctas según el artículo, pero el contenido es de otra
  clase.

# Fase 3 · Clasificación

## 3.1 Separación en entrenamiento y test

Separamos un 20% de los párrafos para el test final, de forma **estratificada** (con la misma
proporción de cada clase en los dos trozos, porque las clases no tienen exactamente el mismo
tamaño). Fijamos la
semilla para que el resultado sea siempre el mismo.

El vectorizador y el modelo los metemos en un `Pipeline` de scikit-learn. Así el vocabulario del
TF-IDF se aprende solo con los textos de entrenamiento y el test no influye en nada.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay

X_train, X_test, y_train, y_test = train_test_split(
    df["text"], df["class"], test_size=0.2, stratify=df["class"], random_state=SEMILLA)

print("Entrenamiento:", len(X_train), " Test:", len(X_test))

## 3.2 Elección del vectorizador y del modelo

Probamos dos formas de vectorizar y tres modelos, todos rápidos y sin GPU:

* **Bag of Words** (`CountVectorizer`): cuántas veces aparece cada lema en el párrafo.
* **TF-IDF**: igual, pero da más peso a los lemas que aparecen en pocos párrafos (`lokomotivo`)
  y menos a los que salen en casi todos (`veturilo`).
* Modelos: **Naive Bayes**, **Regresión Logística** y **SVM lineal**. En los dos últimos usamos
  `class_weight="balanced"` para compensar las diferencias de tamaño entre clases.

Los comparamos con **validación cruzada de 5 particiones** sobre el entrenamiento, usando el
**F1 macro** (la media del F1 de cada clase), que no se deja engañar por la clase mayoritaria.

In [ ]:
modelos = {
    "Naive Bayes": MultinomialNB(),
    "Regresión Logística": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "SVM lineal": LinearSVC(class_weight="balanced", random_state=SEMILLA)
}
vectorizadores = {"Bag of Words": CountVectorizer, "TF-IDF": TfidfVectorizer}
particiones = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEMILLA)

comparacion = []
for nombre_vec, Vectorizador in vectorizadores.items():
    for nombre_mod, modelo in modelos.items():
        pipeline = make_pipeline(Vectorizador(analyzer=lemas_utiles, min_df=2), modelo)
        f1 = cross_val_score(pipeline, X_train, y_train, cv=particiones, scoring="f1_macro")
        comparacion.append({"vectorizador": nombre_vec, "modelo": nombre_mod,
                            "F1 macro (media)": round(f1.mean(), 3), "desviación": round(f1.std(), 3)})

pd.DataFrame(comparacion).sort_values("F1 macro (media)", ascending=False)

Las cuatro mejores combinaciones quedan entre 0,92 y 0,93, con diferencias menores que su
desviación entre particiones (alrededor de 0,02), así que el F1 no basta para decidir.

* **Descartamos Bag of Words + Naive Bayes**, aunque su media sea prácticamente igual, porque en textos
  largos (como las páginas web) sus probabilidades se saturan a 1,0 aunque parte del texto sea de
  otro tema. Naive Bayes suma la evidencia de cada palabra como si fueran independientes, así que con
  cientos de palabras la probabilidad se va al extremo. Lo comprobamos con nuestros datos al final
  de la sección 3.4.
* **Entre los modelos con TF-IDF elegimos la Regresión Logística** (`class_weight="balanced"`),
  porque da probabilidades útiles también en textos largos y sus coeficientes indican qué lemas
  pesan en cada clase (sección *Palabras que más pesan en cada clase*). Además tiene la media más alta
  (0,926), aunque TF-IDF + Naive Bayes y TF-IDF + SVM lineal (0,919) rinden prácticamente igual; la
  elección es por interpretabilidad.

Las diferencias entre modelos son pequeñas: cinco de las seis combinaciones superan un F1 macro de
0,9. La que peor funciona es la SVM con Bag of Words.

## 3.3 Entrenamiento y evaluación en test

In [ ]:
modelo = make_pipeline(
    TfidfVectorizer(analyzer=lemas_utiles, min_df=2),
    LogisticRegression(max_iter=1000, class_weight="balanced"),
)
modelo.fit(X_train, y_train)
y_pred = modelo.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.3f}")
print(f"F1 macro: {f1_score(y_test, y_pred, average='macro'):.3f}\n")
print(classification_report(y_test, y_pred))

ConfusionMatrixDisplay.from_predictions(y_test, y_pred, cmap="Blues")
plt.title("Matriz de confusión (test)")
plt.show()

### Párrafos mal clasificados

Revisar los errores ayuda a entender qué le cuesta al modelo. Muchos son párrafos genéricos que
podrían ser de cualquier tema, es decir, el ruido de etiquetado del que hablábamos en la Fase 1.

In [ ]:
errores = pd.DataFrame({"text": X_test, "real": y_test, "predicho": y_pred})
errores = errores[errores["real"] != errores["predicho"]]
pd.set_option("display.max_colwidth", 150)
errores

### Palabras que más pesan en cada clase

La Regresión Logística aprende un peso para cada lema y clase. Los lemas con más peso son los que
más empujan un texto hacia esa clase.

In [ ]:
vocabulario = modelo[0].get_feature_names_out()
regresion = modelo[1]

for i, clase in enumerate(regresion.classes_):
    mejores = regresion.coef_[i].argsort()[::-1][:12]
    print(f"{clase:<11}", ", ".join(vocabulario[j] for j in mejores))

**Limitación.** Los párrafos de un mismo artículo pueden caer unos en entrenamiento y otros en
test. Como comparten vocabulario muy concreto, el resultado en test probablemente es algo
optimista. Para evitarlo habría que separar por artículo, pero el contrato de datos solo permite
las columnas `text;class`.

## 3.4 Clasificación de páginas web

Descargamos páginas reales en esperanto con `requests`, sacamos el texto de los párrafos (`<p>`)
con BeautifulSoup y le pedimos al modelo la probabilidad de cada clase. Si una página no responde,
se avisa y se sigue con las demás.

In [ ]:
from bs4 import BeautifulSoup

PAGINAS = [
    "https://www.flightschoolusa.com/eo/kiom-gajnas-atc/",
    "https://eo.wikipedia.org/wiki/Transsiberia_fervojo",
    "https://eo.wikipedia.org/wiki/Formulo_1",
]


def texto_de_pagina(url):
    respuesta = requests.get(url, headers={"User-Agent": "practica-mineria-textos-cunef"}, timeout=15)
    respuesta.raise_for_status()
    sopa = BeautifulSoup(respuesta.text, "html.parser")
    for etiqueta in sopa(["script", "style", "nav", "header", "footer", "aside"]):
        etiqueta.decompose()
    parrafos = [limpiar_parrafo(p.get_text(" ")) for p in sopa.find_all("p")]
    return " ".join(p for p in parrafos if len(p) >= LONGITUD_MINIMA)


resultados_web = []
textos_web = {}          # texto de cada página descargada, para la comparación del final de 3.4
for url in PAGINAS:
    try:
        texto = texto_de_pagina(url)
        if not texto:
            raise ValueError("la página no tiene párrafos de texto")
    except Exception as e:
        print(f"No se pudo descargar {url}: {e}")
        continue
    textos_web[url] = texto
    probabilidades = modelo.predict_proba([texto])[0]
    fila = {"página": url, "palabras": len(texto.split())}
    fila.update({clase: round(p, 3) for clase, p in zip(modelo.classes_, probabilidades)})
    fila["predicción"] = modelo.classes_[probabilidades.argmax()]
    resultados_web.append(fila)

pd.DataFrame(resultados_web)

### Textos escritos por nosotros

Por último probamos el modelo con unas frases nuestras. También sirve para comprobar que el
modelo funciona aunque no haya conexión.

In [ ]:
textos_propios = [
    "La trajno alvenis al la stacidomo kun duonhora malfruo pro problemo en la reloj.",
    "Mia nova elektra aŭto bezonas nur unu horon por ŝargi la baterion.",
    "La piloto anoncis, ke la aviadilo surteriĝos en la flughaveno post dek minutoj.",
    "La ŝoforo de la kamiono ne vidis la buson sur la aŭtovojo.",
]

probabilidades = modelo.predict_proba(textos_propios)
tabla = pd.DataFrame(probabilidades.round(3), columns=modelo.classes_)
tabla.insert(0, "texto", textos_propios)
tabla["predicción"] = modelo.predict(textos_propios)
tabla

La frase del coche eléctrico es la que menos convence al modelo. Una posible razón es el lema
`elektra`, que también sale en los párrafos de trenes eléctricos. Contamos en cuántos párrafos de
entrenamiento de cada clase aparece:

In [ ]:
con_elektra = {clase: sum("elektra" in lemas_utiles(t) for t in X_train[y_train == clase])
               for clase in sorted(y_train.unique())}
print('Párrafos de entrenamiento con el lema "elektra":', con_elektra)

### Por qué no Bag of Words + Naive Bayes: probabilidades en textos largos

En la sección 3.2 descartamos Bag of Words + Naive Bayes porque sus probabilidades se saturan en
textos largos. Lo comprobamos con nuestros datos: entrenamos ese modelo con el mismo entrenamiento
y construimos textos largos juntando párrafos del test, 24 de una clase y 8 de las otras (un 75 % y
un 25 %), 5 textos por clase. Comparamos la probabilidad media que da cada modelo a la clase
predicha en los párrafos sueltos y en esos textos largos y, si se han descargado, también en las
páginas web.

In [ ]:
nb_bow = make_pipeline(CountVectorizer(analyzer=lemas_utiles, min_df=2), MultinomialNB())
nb_bow.fit(X_train, y_train)

# Textos largos: 24 párrafos del test de una clase y 8 de las otras, 5 textos por clase
rng = np.random.default_rng(SEMILLA)
test = pd.DataFrame({"text": X_test, "class": y_test})
textos_largos, clases_largos = [], []
for clase in sorted(test["class"].unique()):
    de_la_clase = test.loc[test["class"] == clase, "text"].to_numpy()
    de_otras = test.loc[test["class"] != clase, "text"].to_numpy()
    for _ in range(5):
        partes = list(rng.choice(de_la_clase, 24, replace=False)) + list(rng.choice(de_otras, 8, replace=False))
        textos_largos.append(" ".join(partes))
        clases_largos.append(clase)


def resumen_probabilidades(m):
    # Probabilidad de la clase predicha: media en párrafos sueltos y media y mínimo en textos largos
    sueltos = m.predict_proba(X_test).max(axis=1)
    largos = m.predict_proba(textos_largos).max(axis=1)
    aciertos = (m.predict(textos_largos) == np.array(clases_largos)).sum()
    return {"párrafos sueltos (media)": round(sueltos.mean(), 3),
            "textos largos (media)": round(largos.mean(), 3),
            "textos largos (mínimo)": round(largos.min(), 3),
            "aciertos en textos largos": f"{aciertos} de {len(textos_largos)}"}


print("Probabilidad de la clase predicha")
display(pd.DataFrame({"BoW + Naive Bayes": resumen_probabilidades(nb_bow),
                      "TF-IDF + Regresión Logística": resumen_probabilidades(modelo)}).T)

# Las páginas web, si se pudieron descargar
if textos_web:
    filas = []
    for url, texto in textos_web.items():
        filas.append({"página": url, "palabras": len(texto.split()),
                      "BoW + Naive Bayes": round(nb_bow.predict_proba([texto])[0].max(), 3),
                      "TF-IDF + Regresión Logística": round(modelo.predict_proba([texto])[0].max(), 3)})
    display(pd.DataFrame(filas))
else:
    print("No se descargó ninguna página web, así que no se pueden comparar")

Bag of Words + Naive Bayes da una probabilidad de 1,000 en los 15 textos largos y en las tres
páginas web, aunque en cada texto largo una cuarta parte de los párrafos es de otra clase. La
Regresión Logística acierta los mismos 15 textos con una probabilidad media de 0,943 (mínimo 0,872)
y en las páginas web da entre 0,66 y 0,92, menos en las que mezclan temas. Naive Bayes ya es muy
seguro incluso en párrafos sueltos (0,932 de media, frente a 0,652), así que sus probabilidades no
sirven para saber cuándo un texto es dudoso.

## Conclusiones

### Dataset

* **La Wikipedia necesita mucha limpieza.** Más de la mitad de las líneas que devuelve no son
  párrafos útiles (de 2354 nos quedamos con 981): títulos, elementos de listas, pies de imagen y
  bibliografía. Con unas pocas reglas sencillas (longitud, palabras clave de bibliografía, palabras
  de otros idiomas, empezar en mayúscula, presencia de un verbo conjugado) se quitan casi todas. Cada regla la ajustamos
  revisando lo que dejaba pasar y lo que descartaba por error: por ejemplo, nombres como *Airbus* o
  *Julius* se tomaban por verbos, y párrafos técnicos con unidades (`kg`, `t`) por texto en otro
  idioma.
* **Hacen falta muchos artículos.** Para superar los 300 párrafos por clase después de filtrar
  tuvimos que usar entre 23 y 28 artículos por tema, porque muchos artículos en esperanto son
  cortos.
* **El ruido de etiquetado existe, pero es pequeño.** Cada párrafo hereda la etiqueta de su
  artículo, y no siempre es correcta. Medido con validación cruzada, solo 5 de los 981 párrafos
  contradicen su etiqueta con claridad, y unos 74 (un 7,5 %) si se cuentan también los casos dudosos.
  Son párrafos genéricos, como el de las tiendas de las estaciones, o que hablan de otro tema, como
  los motores de avión dentro del artículo de motores de combustión.

### Modelo de NLP

* **La regularidad del esperanto hace que un sistema de reglas funcione muy bien.** Sin ninguna
  librería de NLP, el modelo acierta el 95,8 % de las categorías gramaticales y el 97,5 % de los
  lemas en frases reales del dataset. En todo el corpus solo deja sin reconocer el 1,2 % de las
  palabras.
* **La lematización es útil para clasificar.** Reduce el vocabulario de 9188 palabras distintas a
  6616 lemas (casi un 28 % menos), porque junta formas como `trajno`, `trajnoj`, `trajnon` y
  `trajnojn`.
* **Lo que más reglas ha necesitado son los nombres propios y las excepciones**, como las palabras
  que parecen participios sin serlo (`konstanta`, `senpilota`). Los errores que quedan en las frases
  reales son nombres propios al inicio de frase (*Harmon*, *Selden*) y palabras cuya categoría
  depende del contexto (*ambaŭ*, *povas*), además de un compuesto con mayúscula (*Esperanto-revuo*)
  que se toma por nombre propio. Algunas reglas son un compromiso: para no estropear nombres como
  *London* o *Washington*, `Konkordon` no pierde el acusativo.
* **Las pruebas evitan romper lo que ya funciona.** Las 36 pruebas de casos concretos detienen el
  cuaderno si alguna falla, así que cada cambio en las reglas se comprueba al momento.

### Clasificación

* **Los temas se distinguen bien a pesar de compartir vocabulario.** El clasificador consigue un F1
  macro de 0,93 en validación cruzada y de 0,944 en test.
* **El F1 no basta para elegir modelo.** Las cuatro mejores combinaciones quedan entre 0,92 y 0,93,
  con diferencias menores que la desviación entre particiones. Descartamos Bag of Words + Naive
  Bayes porque sus probabilidades se saturan en textos largos: en nuestra comprobación da 1,000 en
  los 15 textos largos y en las tres páginas web, mientras que la Regresión Logística da 0,943 de
  media en los textos largos (mínimo 0,872). Elegimos la Regresión Logística también porque sus
  coeficientes se pueden interpretar, y los lemas que más pesan son los que una persona esperaría
  (`trajno`, `aviadilo`, `aŭto`...).
* **Los errores van sobre todo hacia `automobilo`.** 5 de los 11 errores en test son párrafos de
  otra clase que el modelo toma por coches (otros 4 los toma por aviación). Comparte con las otras
  dos clases palabras como `motoro`, `veturilo` o `elektra`; por ejemplo, el lema `elektra` aparece
  en 24 párrafos de entrenamiento de `fervojo` y en otros 24 de `automobilo`, lo que explica que la
  frase del coche eléctrico sea la más dudosa.
* **Generaliza fuera de la Wikipedia.** Acierta las tres páginas web, incluida una que no es de la
  Wikipedia, y las cuatro frases escritas por nosotros, que tienen un estilo menos enciclopédico. La
  confianza baja en páginas largas que mezclan temas: un 66 % en la de Fórmula 1 frente a un 92 % en
  la del Transiberiano.
* **Ampliar el dataset hizo que el modelo esté mucho más seguro con las páginas web**: en la página
  de aviación pasó de un 45 % a un 70 % de probabilidad para la clase correcta.

### Limitaciones

* **El resultado en test probablemente sea algo optimista.** Párrafos de un mismo artículo pueden
  acabar unos en entrenamiento y otros en test, y comparten vocabulario muy concreto. Separar por
  artículo no es posible porque el contrato de datos solo admite las columnas `text;class`.
* **Las páginas web se clasifican enteras.** El modelo se entrenó con párrafos de unas 30 palabras,
  pero cada página se clasifica como un único texto de cientos o miles de palabras. Eso explica en
  parte la menor confianza en las páginas largas.
* **El dataset solo tiene textos de la Wikipedia.** El estilo es enciclopédico, y no se ha probado
  con textos de otro tipo, como noticias o foros, más allá de unas pocas frases.

### Posibles mejoras

* Guardar internamente el artículo de cada párrafo, sin cambiar el CSV, y usar validación cruzada
  por artículo (`GroupKFold`) para medir cuánto infla el resultado la fuga entre entrenamiento y
  test.
* Clasificar las páginas web párrafo a párrafo y promediar las probabilidades, igual que se entrenó
  el modelo.
* Usar la medida del ruido de etiquetado para revisar a mano, o descartar, los párrafos que
  contradicen su etiqueta con claridad.
* Ampliar el dataset con más artículos y otras fuentes en esperanto.